# **Decision Tree Classifier**

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

spotify = pd.read_csv('spotify_clean.csv', index_col=0)
spotify.reset_index(inplace=True)

spotify['popularity_flag'] = (spotify['popularity'] >= spotify['popularity'].describe()["75%"]).astype(int)

## Sampling
majority = spotify[spotify.popularity_flag == 0]
minority = spotify[spotify.popularity_flag == 1]

majority_undersampled = resample(majority,
replace=False,
n_samples=len(spotify[spotify["popularity_flag"] == 1]),
random_state=33)

spotify = pd.concat([majority_undersampled, minority])

print(spotify['popularity_flag'].value_counts())

## Defining X and Y
X = spotify.drop(columns=["popularity","popularity_flag"])
y = spotify["popularity_flag"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

popularity_flag
0    29367
1    29367
Name: count, dtype: int64


In [3]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

# Set up GridSearchCV
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=33),
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring="f1",
    n_jobs=-1
)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

# Best Model
best_dt_clf = grid_search.best_estimator_
y_pred = best_dt_clf.predict(X_test)

# Compute Classification Metrics
accuracy = accuracy_score(y_test, y_pred)
print("Best hyperparameters:", grid_search.best_params_)
print(f"Test Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

Best hyperparameters: {'criterion': 'gini', 'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2}
Test Accuracy: 0.5693
              precision    recall  f1-score   support

           0       0.63      0.31      0.41      5806
           1       0.55      0.83      0.66      5941

    accuracy                           0.57     11747
   macro avg       0.59      0.57      0.54     11747
weighted avg       0.59      0.57      0.54     11747



In [5]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, digits=4))


              precision    recall  f1-score   support

           0     0.6327    0.3062    0.4127      5806
           1     0.5493    0.8263    0.6599      5941

    accuracy                         0.5693     11747
   macro avg     0.5910    0.5663    0.5363     11747
weighted avg     0.5905    0.5693    0.5377     11747

